In [ ]:
import re
import json
import time
import joblib
import random
import requests
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from sklearn.metrics import f1_score
import torch

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
val_df = pd.read_pickle("val_df.pkl")
test_df = pd.read_pickle("test_df.pkl")

Y_val = joblib.load("Y_val.pkl")
Y_test = joblib.load("Y_test.pkl")

mlb = joblib.load("label_binarizer.pkl")
label_list = list(mlb.classes_)
label_set = set(label_list)

print("val_df:", val_df.shape)
print("test_df:", test_df.shape)
print("Y_val:", Y_val.shape)
print("Y_test:", Y_test.shape)
print("Num labels:", len(label_list))
print("Sample labels:", label_list[:10])

assert "text" in val_df.columns, "val_df must contain a 'text' column"
assert "text" in test_df.columns, "test_df must contain a 'text' column"

In [ ]:
OLLAMA_HOST = "http://warhol.informatik.rwth-aachen.de:11434"

# Replace with exact model names from /api/tags if needed
OLLAMA_MODELS = [
    "gpt-oss:20b",
    "gemma3:27b",
    "llama3.3:70b",
    "codellama:latest",
    "deepseek-r1:32b",
    "deepseek-r1:7b" ,# weak SLM baseline, but available
]

# Start small, then increase
N_VAL = 50      # validation subset for quick comparison
N_TEST = 50     # test subset later, only after validation looks sane

MAX_RETURN_CODES = 5
REQUEST_TIMEOUT = 300
TEMPERATURE = 0.0
TOP_K = 5

In [ ]:
SYSTEM_PROMPT = """You are a clinical coding assistant.

Task:
Predict diagnosis ICD codes from a discharge summary.

Rules:
1. Only choose codes from the provided valid ICD code list.
2. Return between 1 and 5 codes.
3. Rank codes from most likely to least likely.
4. Do not invent codes.
5. Return JSON only.
"""

EXAMPLE_BLOCK = """
Example:
Discharge Summary:
\"\"\"
Patient with hypertension and type 2 diabetes mellitus admitted for uncontrolled blood pressure and hyperglycemia. Discharged on antihypertensives and diabetic medications.
\"\"\"

Valid ICD Codes:
I10, E119, J189, N179

Correct JSON:
{"predicted_codes": ["I10", "E119"]}
"""

def build_prompt(note_text, label_list):
    valid_codes = ", ".join(label_list)
    return f"""
{EXAMPLE_BLOCK}

Discharge Summary:
\"\"\"
{note_text}
\"\"\"

Valid ICD Codes:
{valid_codes}

Return only valid JSON:
{{"predicted_codes": ["CODE1", "CODE2"]}}
"""

In [ ]:
JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "predicted_codes": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": MAX_RETURN_CODES
        }
    },
    "required": ["predicted_codes"]
}

def clean_predicted_codes(codes, label_set, max_codes=MAX_RETURN_CODES):
    if not isinstance(codes, list):
        return []
    cleaned = []
    for c in codes:
        if isinstance(c, str):
            c = c.strip()
            if c in label_set and c not in cleaned:
                cleaned.append(c)
        if len(cleaned) >= max_codes:
            break
    return cleaned

def parse_predicted_codes(raw_text, label_set, max_codes=MAX_RETURN_CODES):
    raw_text = str(raw_text).strip()

    # Direct JSON parse
    try:
        obj = json.loads(raw_text)
        return clean_predicted_codes(obj.get("predicted_codes", []), label_set, max_codes)
    except Exception:
        pass

    # Extract first JSON object if wrapped in extra text
    start = raw_text.find("{")
    end = raw_text.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = raw_text[start:end+1]
        try:
            obj = json.loads(candidate)
            return clean_predicted_codes(obj.get("predicted_codes", []), label_set, max_codes)
        except Exception:
            pass

    # Fallback: regex for label-like tokens already in label_set
    found = []
    for token in re.findall(r"[A-Za-z0-9\.]+", raw_text):
        if token in label_set and token not in found:
            found.append(token)
        if len(found) >= max_codes:
            break
    return found

In [ ]:
def ollama_chat_json(model_name, note_text, label_list, temperature=TEMPERATURE):
    payload = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(note_text, label_list)}
        ],
        "format": JSON_SCHEMA,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }

    r = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        headers={"Content-Type": "application/json"},
        data=json.dumps(payload),
        timeout=REQUEST_TIMEOUT
    )
    r.raise_for_status()
    data = r.json()
    return data["message"]["content"]

def predict_codes_ollama(model_name, note_text):
    try:
        raw = ollama_chat_json(model_name, note_text, label_list)
        return parse_predicted_codes(raw, label_set)
    except Exception as e:
        print(f"[{model_name}] error:", e)
        return []

In [ ]:
def evaluate_code_lists(y_true_bin, pred_bin):
    micro = f1_score(y_true_bin, pred_bin, average="micro", zero_division=0)
    macro = f1_score(y_true_bin, pred_bin, average="macro", zero_division=0)
    return {"micro_f1": float(micro), "macro_f1": float(macro)}

def precision_at_k_from_lists(true_bin, pred_lists, mlb, k=TOP_K):
    true_sets = [set(mlb.classes_[np.where(row == 1)[0]]) for row in true_bin]
    vals = []
    for true_set, pred in zip(true_sets, pred_lists):
        pred_set = set(pred[:k])
        vals.append(len(true_set & pred_set) / float(k))
    return float(np.mean(vals)) if vals else 0.0

def recall_at_k_from_lists(true_bin, pred_lists, mlb, k=TOP_K):
    true_sets = [set(mlb.classes_[np.where(row == 1)[0]]) for row in true_bin]
    vals = []
    for true_set, pred in zip(true_sets, pred_lists):
        if len(true_set) == 0:
            continue
        pred_set = set(pred[:k])
        vals.append(len(true_set & pred_set) / float(len(true_set)))
    return float(np.mean(vals)) if vals else 0.0

def empty_rate(pred_lists):
    return float(sum(len(x) == 0 for x in pred_lists) / len(pred_lists)) if pred_lists else 0.0

def avg_num_codes(pred_lists):
    return float(np.mean([len(x) for x in pred_lists])) if pred_lists else 0.0

In [ ]:
def run_model_on_subset(model_name, predictor_fn, subset_df, sleep_sec=0.0):
    preds = []
    times = []

    for text in tqdm(subset_df["text"].tolist(), desc=model_name):
        t0 = time.time()
        codes = predictor_fn(text)
        dt = time.time() - t0

        preds.append(codes)
        times.append(dt)

        if sleep_sec > 0:
            time.sleep(sleep_sec)

    return preds, times

In [ ]:
val_subset = val_df.iloc[:N_VAL].copy()
Y_val_subset = Y_val[:N_VAL]

results_rows = []

# Ollama models
for model_name in OLLAMA_MODELS:
    preds, times = run_model_on_subset(
        model_name=model_name,
        predictor_fn=lambda text, m=model_name: predict_codes_ollama(m, text),
        subset_df=val_subset
    )

    pred_bin = mlb.transform(preds)
    metrics = evaluate_code_lists(Y_val_subset, pred_bin)

    results_rows.append({
        "model": model_name,
        "source": "ollama_api",
        "mode": "direct",
        "split": "val",
        "n": len(val_subset),
        "micro_f1": metrics["micro_f1"],
        "macro_f1": metrics["macro_f1"],
        "precision@5": precision_at_k_from_lists(Y_val_subset, preds, mlb, k=5),
        "recall@5": recall_at_k_from_lists(Y_val_subset, preds, mlb, k=5),
        "avg_latency_sec": float(np.mean(times)),
        "empty_rate": empty_rate(preds),
        "avg_num_codes": avg_num_codes(preds)
    })


results_df = pd.DataFrame(results_rows).sort_values(["micro_f1", "macro_f1"], ascending=False)
results_df

In [ ]:
results_df.to_csv("results/llm_direct_validation_comparison.csv", index=False)
print("Saved: llm_direct_validation_comparison.csv")

## Multi-Role

In [ ]:
import os
import re
import gc
import json
import time
import math
import joblib
import random
import requests
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
OLLAMA_HOST = "http://warhol.informatik.rwth-aachen.de:11434"

OLLAMA_MODELS = [
    "gpt-oss:20b",
    "gemma3:27b",
    "llama3.3:70b",
    "codellama:latest",
    "deepseek-r1:32b",
    "deepseek-r1:7b" ,
]

N_VAL = 50          # start with 20 if it is too slow
N_TEST = 50         # use later
MAX_RETURN_CODES = 5
MAX_PROPOSAL_CODES = 8
REQUEST_TIMEOUT = 300
TEMPERATURE = 0.0
TOP_K = 5
RUN_TEST = False    # set True only after validation looks okay

In [ ]:
def list_ollama_models():
    r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=30)
    r.raise_for_status()
    data = r.json()
    return data.get("models", [])

try:
    models_info = list_ollama_models()
    available_names = [m["name"] for m in models_info]
    print("Available Ollama models:")
    for name in available_names:
        print("-", name)
except Exception as e:
    print("Could not fetch /api/tags:", e)
    available_names = []

In [ ]:
def clean_predicted_codes(codes, label_set, max_codes=MAX_RETURN_CODES):
    if not isinstance(codes, list):
        return []
    cleaned = []
    for c in codes:
        if isinstance(c, str):
            c = c.strip()
            if c in label_set and c not in cleaned:
                cleaned.append(c)
        if len(cleaned) >= max_codes:
            break
    return cleaned

def parse_json_field(raw_text, field_name, allowed_set, max_codes=MAX_RETURN_CODES):
    raw_text = str(raw_text).strip()

    # direct parse
    try:
        obj = json.loads(raw_text)
        return clean_predicted_codes(obj.get(field_name, []), allowed_set, max_codes)
    except Exception:
        pass

    # extract JSON block
    start = raw_text.find("{")
    end = raw_text.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            obj = json.loads(raw_text[start:end+1])
            return clean_predicted_codes(obj.get(field_name, []), allowed_set, max_codes)
        except Exception:
            pass

    # fallback regex
    found = []
    for token in re.findall(r"[A-Za-z0-9\.]+", raw_text):
        if token in allowed_set and token not in found:
            found.append(token)
        if len(found) >= max_codes:
            break
    return found

In [ ]:
def ollama_chat_json(model_name, system_prompt, user_prompt, schema, temperature=0):
    payload = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "format": schema,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }

    r = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        headers={"Content-Type": "application/json"},
        data=json.dumps(payload),
        timeout=REQUEST_TIMEOUT
    )
    r.raise_for_status()
    data = r.json()
    return data["message"]["content"]

In [ ]:
def call_model_json(model_name, system_prompt, user_prompt, schema):
    return ollama_chat_json(
            model_name=model_name,
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            schema=schema
        )

In [ ]:
CLINICIAN_SCHEMA = {
    "type": "object",
    "properties": {
        "proposed_codes": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": MAX_PROPOSAL_CODES
        }
    },
    "required": ["proposed_codes"]
}

CLINICIAN_SYSTEM = """You are a senior physician reviewing a discharge summary.

Task:
Identify the diagnosis ICD codes that are clinically supported by the note.

Rules:
1. Use only codes from the provided valid ICD code list.
2. Propose up to 8 likely codes.
3. It is better to include plausible candidates here, but do not invent unsupported diagnoses.
4. Return JSON only.

Required JSON format:
{"proposed_codes": ["CODE1", "CODE2"]}
"""

In [ ]:
CODER_SCHEMA = {
    "type": "object",
    "properties": {
        "ranked_codes": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": MAX_RETURN_CODES
        }
    },
    "required": ["ranked_codes"]
}

CODER_SYSTEM = """You are an experienced hospital clinical coder.

Task:
Select and rank the most appropriate diagnosis ICD codes.

Rules:
1. Use only codes from the provided candidate list.
2. Choose between 1 and 5 codes.
3. Be precise and conservative.
4. Rank codes from most likely to least likely.
5. Return JSON only.

Required JSON format:
{"ranked_codes": ["CODE1", "CODE2"]}
"""

In [ ]:
AUDITOR_SCHEMA = {
    "type": "object",
    "properties": {
        "verified_codes": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": MAX_RETURN_CODES
        }
    },
    "required": ["verified_codes"]
}

AUDITOR_SYSTEM = """You are a strict clinical documentation auditor.

Task:
Verify whether the proposed ICD diagnosis codes are directly supported by the discharge summary.

Rules:
1. Use only codes from the proposed code list.
2. Remove unsupported or weakly supported codes.
3. Keep only the final verified codes.
4. Return between 1 and 5 codes.
5. Return JSON only.

Required JSON format:
{"verified_codes": ["CODE1", "CODE2"]}
"""

In [ ]:
def build_clinician_prompt(note_text, label_list):
    valid_codes = ", ".join(label_list)
    return f"""
Discharge Summary:
\"\"\"
{note_text}
\"\"\"

Valid ICD Codes:
{valid_codes}

Instructions:
- Identify the diagnoses strongly or reasonably supported by the discharge summary.
- Return up to 8 codes.
- Return only valid JSON.

JSON:
{{"proposed_codes": ["CODE1", "CODE2"]}}
"""

def build_coder_prompt(note_text, proposed_codes):
    candidate_str = ", ".join(proposed_codes)
    return f"""
Discharge Summary:
\"\"\"
{note_text}
\"\"\"

Candidate ICD Codes proposed by clinician:
{candidate_str}

Instructions:
- Select the best final diagnosis codes from this candidate list only.
- Be conservative.
- Return 1 to 5 codes ranked by confidence.
- Return only valid JSON.

JSON:
{{"ranked_codes": ["CODE1", "CODE2"]}}
"""

def build_auditor_prompt(note_text, ranked_codes):
    ranked_str = ", ".join(ranked_codes)
    return f"""
Discharge Summary:
\"\"\"
{note_text}
\"\"\"

Proposed final ICD Codes:
{ranked_str}

Instructions:
- Verify which of these are directly supported by the discharge summary.
- Remove unsupported codes.
- Return 1 to 5 verified codes only.
- Return only valid JSON.

JSON:
{{"verified_codes": ["CODE1", "CODE2"]}}
"""

In [ ]:
def predict_codes_three_role(model_name, note_text):
    try:
        # Role 1: Clinician
        raw1 = call_model_json(
            model_name=model_name,
            system_prompt=CLINICIAN_SYSTEM,
            user_prompt=build_clinician_prompt(note_text, label_list),
            schema=CLINICIAN_SCHEMA
        )
        proposed_codes = parse_json_field(
            raw_text=raw1,
            field_name="proposed_codes",
            allowed_set=label_set,
            max_codes=MAX_PROPOSAL_CODES
        )

        if len(proposed_codes) == 0:
            return []

        # Role 2: Coder
        raw2 = call_model_json(
            model_name=model_name,
            system_prompt=CODER_SYSTEM,
            user_prompt=build_coder_prompt(note_text, proposed_codes),
            schema=CODER_SCHEMA
        )
        ranked_codes = parse_json_field(
            raw_text=raw2,
            field_name="ranked_codes",
            allowed_set=set(proposed_codes),
            max_codes=MAX_RETURN_CODES
        )

        if len(ranked_codes) == 0:
            return proposed_codes[:MAX_RETURN_CODES]

        # Role 3: Auditor
        raw3 = call_model_json(
            model_name=model_name,
            system_prompt=AUDITOR_SYSTEM,
            user_prompt=build_auditor_prompt(note_text, ranked_codes),
            schema=AUDITOR_SCHEMA
        )
        verified_codes = parse_json_field(
            raw_text=raw3,
            field_name="verified_codes",
            allowed_set=set(ranked_codes),
            max_codes=MAX_RETURN_CODES
        )

        if len(verified_codes) == 0:
            return ranked_codes[:MAX_RETURN_CODES]

        return verified_codes[:MAX_RETURN_CODES]

    except Exception as e:
        print(f"[{model_name}] three-role error:", e)
        return []

In [ ]:
def evaluate_code_lists(y_true_bin, pred_bin):
    micro = f1_score(y_true_bin, pred_bin, average="micro", zero_division=0)
    macro = f1_score(y_true_bin, pred_bin, average="macro", zero_division=0)
    return {"micro_f1": float(micro), "macro_f1": float(macro)}

def precision_at_k_from_lists(true_bin, pred_lists, mlb, k=TOP_K):
    true_sets = [set(mlb.classes_[np.where(row == 1)[0]]) for row in true_bin]
    vals = []
    for true_set, pred in zip(true_sets, pred_lists):
        pred_set = set(pred[:k])
        vals.append(len(true_set & pred_set) / float(k))
    return float(np.mean(vals)) if vals else 0.0

def recall_at_k_from_lists(true_bin, pred_lists, mlb, k=TOP_K):
    true_sets = [set(mlb.classes_[np.where(row == 1)[0]]) for row in true_bin]
    vals = []
    for true_set, pred in zip(true_sets, pred_lists):
        if len(true_set) == 0:
            continue
        pred_set = set(pred[:k])
        vals.append(len(true_set & pred_set) / float(len(true_set)))
    return float(np.mean(vals)) if vals else 0.0

def empty_rate(pred_lists):
    return float(sum(len(x) == 0 for x in pred_lists) / len(pred_lists)) if pred_lists else 0.0

def avg_num_codes(pred_lists):
    return float(np.mean([len(x) for x in pred_lists])) if pred_lists else 0.0

In [ ]:
def run_model_on_subset(model_name, predictor_fn, subset_df):
    preds = []
    times = []

    for text in tqdm(subset_df["text"].tolist(), desc=model_name):
        t0 = time.time()
        codes = predictor_fn(text)
        dt = time.time() - t0

        preds.append(codes)
        times.append(dt)

    return preds, times

In [ ]:
val_subset = val_df.iloc[:N_VAL].copy()
Y_val_subset = Y_val[:N_VAL]

results_rows = []
predictions_cache = {}

In [ ]:
for model_name in OLLAMA_MODELS:
    preds, times = run_model_on_subset(
        model_name=model_name,
        predictor_fn=lambda text, m=model_name: predict_codes_three_role(m, text),
        subset_df=val_subset
    )

    pred_bin = mlb.transform(preds)
    metrics = evaluate_code_lists(Y_val_subset, pred_bin)

    predictions_cache[(model_name, "val", "three_role")] = preds

    results_rows.append({
        "model": model_name,
        "source": "ollama_api",
        "mode": "three_role",
        "split": "val",
        "n": len(val_subset),
        "micro_f1": metrics["micro_f1"],
        "macro_f1": metrics["macro_f1"],
        "precision@5": precision_at_k_from_lists(Y_val_subset, preds, mlb, k=5),
        "recall@5": recall_at_k_from_lists(Y_val_subset, preds, mlb, k=5),
        "avg_latency_sec": float(np.mean(times)),
        "empty_rate": empty_rate(preds),
        "avg_num_codes": avg_num_codes(preds)
    })

In [ ]:
results_df = pd.DataFrame(results_rows).sort_values(
    ["micro_f1", "macro_f1"],
    ascending=False
)

results_df

In [ ]:
results_df.to_csv("results/three_role_validation_comparison.csv", index=False)
print("Saved: three_role_validation_comparison.csv")

In [ ]:
def inspect_predictions(subset_df, y_true, pred_lists, model_name, n=5):
    true_lists = [list(mlb.classes_[np.where(row == 1)[0]]) for row in y_true]
    for i in range(min(n, len(subset_df))):
        print("=" * 100)
        print("MODEL:", model_name)
        print("TEXT:", subset_df["text"].iloc[i][:700], "...")
        print("TRUE:", true_lists[i])
        print("PRED:", pred_lists[i])

# Example:
inspect_predictions(val_subset, Y_val_subset, predictions_cache[("gpt-oss:20b", "val", "three_role")], "gpt-oss:20b", n=3)

In [ ]:
if RUN_TEST:
    test_subset = test_df.iloc[:N_TEST].copy()
    Y_test_subset = Y_test[:N_TEST]

    test_rows = []

    for model_name in OLLAMA_MODELS:
        preds, times = run_model_on_subset(
            model_name=model_name,
            predictor_fn=lambda text, m=model_name: predict_codes_three_role(m, text),
            subset_df=test_subset
        )

        pred_bin = mlb.transform(preds)
        metrics = evaluate_code_lists(Y_test_subset, pred_bin)

        test_rows.append({
            "model": model_name,
            "source": "ollama_api",
            "mode": "three_role",
            "split": "test",
            "n": len(test_subset),
            "micro_f1": metrics["micro_f1"],
            "macro_f1": metrics["macro_f1"],
            "precision@5": precision_at_k_from_lists(Y_test_subset, preds, mlb, k=5),
            "recall@5": recall_at_k_from_lists(Y_test_subset, preds, mlb, k=5),
            "avg_latency_sec": float(np.mean(times)),
            "empty_rate": empty_rate(preds),
            "avg_num_codes": avg_num_codes(preds)
        })

    test_results_df = pd.DataFrame(test_rows).sort_values(
        ["micro_f1", "macro_f1"],
        ascending=False
    )

    test_results_df.to_csv("results/three_role_test_comparison.csv", index=False)
    test_results_df

## Chain-Of-Thought Prompting

In [ ]:
COT_REASONER_SCHEMA = {
    "type": "object",
    "properties": {
        "diagnoses": {
            "type": "array",
            "items": {"type": "string"}
        }
    },
    "required": ["diagnoses"]
}

COT_MAPPER_SCHEMA = {
    "type": "object",
    "properties": {
        "predicted_codes": {
            "type": "array",
            "items": {"type": "string"}
        }
    },
    "required": ["predicted_codes"]
}

In [ ]:
def build_cot_reasoner_prompt(note_text):
    return f"""
Read the discharge summary and identify the main diagnoses, comorbidities, and clinically relevant conditions.

Discharge Summary:
{note_text}

Return only JSON with:
- diagnoses: a list of short diagnosis phrases
"""

In [ ]:
def build_cot_mapper_prompt(diagnoses, label_list):
    valid_codes = ", ".join(label_list)
    diag_text = ", ".join(diagnoses)

    return f"""
You are given extracted diagnoses from a discharge summary.

Diagnoses:
{diag_text}

Valid ICD codes:
{valid_codes}

Select the most relevant ICD codes from the valid list only.
Return at most 5 codes.

Return only JSON with:
- predicted_codes: list of ICD codes
"""

In [ ]:
def predict_codes_cot(model_name, note_text):
    try:
        raw1 = call_model_json(
            model_name=model_name,
            system_prompt="You extract structured clinical diagnoses from discharge summaries.",
            user_prompt=build_cot_reasoner_prompt(note_text),
            schema=COT_REASONER_SCHEMA
        )
        diagnoses = raw1.get("diagnoses", [])
        if not diagnoses:
            return []

        raw2 = call_model_json(
            model_name=model_name,
            system_prompt="You map diagnoses to valid ICD codes.",
            user_prompt=build_cot_mapper_prompt(diagnoses, label_list),
            schema=COT_MAPPER_SCHEMA
        )
        return clean_predicted_codes(raw2.get("predicted_codes", []), label_set)

    except Exception as e:
        print(f"[COT ERROR] {model_name}: {e}")
        return []

In [ ]:
for model_name in OLLAMA_MODELS:
    preds, times = run_model_on_subset(
        model_name=model_name,
        predictor_fn=lambda text, m=model_name: predict_codes_cot(m, text),
        subset_df=val_subset
    )

    pred_bin = mlb.transform(preds)
    metrics = evaluate_code_lists(Y_val_subset, pred_bin)

    results_rows.append({
        "model": model_name,
        "method": "cot",
        "micro_f1": metrics["micro_f1"],
        "macro_f1": metrics["macro_f1"],
        "p@5": precision_at_k_from_lists(Y_val_subset, preds, mlb),
        "r@5": recall_at_k_from_lists(Y_val_subset, preds, mlb),
        "avg_latency_sec": float(np.mean(times)),
        "empty_rate": empty_rate(preds)
    })

In [ ]:
results_df